# LangChain 核心模块 Agent - OpenAI Function


In [1]:
%%capture --no-stderr
# 去掉pip install 使用项目依赖版本
# %pip install -U langchain

In [2]:
from langchain_openai import ChatOpenAI

# 使用 gpt-4o-mini
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
from langchain.agents import tool

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

tools = [get_word_length]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = "你是非常强大的AI助手，但在计算单词长度方面不擅长。"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")
])

In [ ]:
from langchain.agents import create_openai_functions_agent

agent = create_openai_functions_agent(chat_model, tools, prompt)

In [6]:
from langchain.agents import AgentExecutor

# 实例化 OpenAIFunctionsAgent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [7]:
agent_executor.invoke("单词“educa”中有多少个字母?")



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'educa'}`


5单词“educa”中有5个字母。

> Finished chain.


{'input': '单词“educa”中有多少个字母?', 'output': '单词“educa”中有5个字母。'}

In [ ]:
MEMORY_KEY = "chat_history"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name=MEMORY_KEY),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")
])

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 创建会话历史存储
chat_history_store = {}

def get_session_history(session_id: str):
    """根据 session_id 获取或创建会话历史"""
    if session_id not in chat_history_store:
        chat_history_store[session_id] = ChatMessageHistory()
    return chat_history_store[session_id]

In [ ]:
agent = create_openai_functions_agent(chat_model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 使用 RunnableWithMessageHistory 包装 agent_executor 以支持会话历史
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key=MEMORY_KEY,
)

In [ ]:
agent_with_chat_history.invoke(
    {"input": "单词"educa"中有多少个字母?"},
    config={"configurable": {"session_id": "session1"}}
)

In [ ]:
# 继续对话，使用相同的 session_id 以保持上下文
agent_with_chat_history.invoke(
    {"input": "那是一个真实的单词吗？"},
    config={"configurable": {"session_id": "session1"}}
)